主要实现内容：合并基于近15天销量、近30天销量、运营提供的参数生成的发货建议

In [1]:
import pandas as pd

In [2]:
parms = '20261109'
send_goods15 = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-{parms}-15.xlsx')
send_goods30 = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-{parms}-30.xlsx')
send_goods60 = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-{parms}-60.xlsx')
send_goods = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-模板.xlsx')
send_goods_ca = pd.read_excel(f'../程序建议发货列表/发货列表{parms}-new-7.0-120-模板-CA.xlsx')
shipment_path = f'../src_data/在途货件/在途货件{parms}-新版.xlsx'

In [3]:
send_goods_ca.columns

Index(['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数',
       '快递打包时间', '空运打包时间', '海运打包时间', '补货', '借调', 'FBA在库可售天数', 'FBA总可售天数',
       '本地库存可售天数', '总库存可售天数', '预估在库日均', '断货风险总天数', '断货总损失销量', '首次断货前可售天数',
       'FBA库存', '已出运', 'FBA预占', 'FBA在途', '差异量-已计入可用库存', 'dhl_pre', 'air_pre',
       'sea_pre', 'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数',
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', '预占总数', '快递_预占', '空运_预占',
       '海运_预占', 'FBA自提物流_预占', 'CA', 'DE', 'JP', 'UK', 'US', 'TK本地仓', '共享',
       '本地-在途', '已下单数量', '已生产未发货', 'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓',
       'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓',
       'IT永翔海外仓', 'CN易速达:易速达美东GA仓', '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓',
       '顺丰SF:美国芝加哥S06仓', '7天日均', '15天日均', '30天日均', '断货时间1', '断货总天数1', '损失销量1',
       '断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', '断货时间3', '断货总天数3',
       '损失销量3', '断货开

In [4]:
# 筛选 不计入可售的差异量
shipment_receive_total = pd.read_excel(shipment_path, sheet_name='接收中-汇总')
shipment_receive_total_no = shipment_receive_total[shipment_receive_total['是否已计入可售'] == '否'].copy()

In [5]:
# shipment_receive_total_no.to_excel(f'D:/shipping_recommendations2.0/src_data/在途货件/在途货件{parms}-shipment_receive_total_no.xlsx')

In [6]:
shipment_receive_detail = pd.read_excel(shipment_path, sheet_name='接收中-货件详情')

future_records_detail = pd.read_excel(shipment_path, sheet_name='预计到仓日期早于当天日期的记录')
future_records_total = pd.read_excel(shipment_path, sheet_name='预计到仓日期早于当天日期的记录-汇总')

inspection_excluded_df = pd.read_excel(shipment_path, sheet_name='不计入在途的查验货件详情')
inspection_excluded_total = pd.read_excel(shipment_path, sheet_name='不计入在途的查验货件MSKU汇总')
inspection_excluded_total['仓库'] = inspection_excluded_total['仓库'].str.replace('_FBA','')
inspection_excluded_total = inspection_excluded_total.rename(columns={'仓库': '店铺-站点', '发货量':'不计入在途的查验量'})
# 删除不需要的列
inspection_excluded_total = inspection_excluded_total.drop(columns=['SKU'])

In [7]:
inspection_excluded_total

,店铺-站点,MSKU,不计入在途的查验量
0,HK_SIMARI_Limited:US,US-RG805-Black-2XL,45
1,HK_SIMARI_Limited:US,US-RG805-Black-L,180
2,HK_SIMARI_Limited:US,US-RG805-Black-M,160
3,HK_SIMARI_Limited:US,US-RG805-Black-S,60
4,HK_SIMARI_Limited:US,US-RG805-Black-XL,85
5,HK_SIMARI_Limited:US,US-RG805-Caramel-S,5
6,HK_SIMARI_Limited:US,US-RG805-Grey-L,10
7,HK_SIMARI_Limited:US,US-RG805-Grey-M,5
8,HK_SIMARI_Limited:US,US-RG805-Grey-S,5
9,HK_SIMARI_Limited:US,US-RG805-Grey-XL,5


In [8]:
send_goods.columns

Index(['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数',
       '快递打包时间', '空运打包时间', '海运打包时间', '补货', '借调', 'FBA在库可售天数', 'FBA总可售天数',
       '本地库存可售天数', '总库存可售天数', '预估在库日均', '断货风险总天数', '断货总损失销量', '首次断货前可售天数',
       'FBA库存', '已出运', 'FBA预占', 'FBA在途', '差异量-已计入可用库存', 'dhl_pre', 'air_pre',
       'sea_pre', 'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数',
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', '预占总数', '快递_预占', '空运_预占',
       '海运_预占', 'FBA自提物流_预占', 'CA', 'DE', 'JP', 'UK', 'US', 'TK本地仓', '共享',
       '本地-在途', '已下单数量', '已生产未发货', 'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓',
       'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓',
       'IT永翔海外仓', 'CN易速达:易速达美东GA仓', '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓',
       '顺丰SF:美国芝加哥S06仓', '7天日均', '15天日均', '30天日均', '断货时间1', '断货总天数1', '损失销量1',
       '断货开始天数1', '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', '断货时间3', '断货总天数3',
       '损失销量3', '断货开

In [9]:
select_columns = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型', '规模定位', '需求定位', '发货定位', '备货定位', '补货', '借调', 
                  'FBA在库可售天数', 'FBA总可售天数', '预估在库日均','断货风险总天数', '首次断货前可售天数', 
                  '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
                  # 'FBA库存', '已出运','FBA预占','FBA在途', '预占总数', '快递_预占', '空运_预占', '海运_预占', 
                  'FBA库存', '已出运','FBA预占','FBA在途', '差异量-已计入可用库存','预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占', 
                  'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数', 'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 
                  'CA', 'DE', 'JP', 'UK', 'US','TK本地仓', '共享', '本地-在途', '已下单数量', '已生产未发货',
                  'CA仓搜海外仓','DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
                  # '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
                            '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国芝加哥S06仓',
                  '7天日均', '15天日均', '30天日均',
                  '断货时间1', '断货总天数1', '损失销量1', '断货开始天数1',
                  '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', 
                  '断货时间3', '断货总天数3', '损失销量3', '断货开始天数3'
                 ]

# '断货时间4', '断货总天数4','损失销量4', '断货开始天数4'
# '断货时间5', '断货总天数5','损失销量5', '断货开始天数5'
send_goods = send_goods[select_columns]

select_columns_ca = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数',
       '快递打包时间', '空运打包时间', '海运打包时间', '补货', '借调', 'FBA在库可售天数', 'FBA总可售天数',
       '本地库存可售天数', '总库存可售天数', '预估在库日均', '断货风险总天数', '断货总损失销量', '首次断货前可售天数',
       'FBA库存', '已出运', 'FBA预占', 'FBA在途', '差异量-已计入可用库存', 'dhl_pre', 'air_pre', 'sea_pre',
       'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数', 'dhl_pre缺货数',
       'air_pre缺货数', 'sea_pre缺货数', '预占总数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占', 'CA', 'DE', 'JP', 'UK', 'US', 'TK本地仓', '共享', '本地-在途',
       '已下单数量', '已生产未发货', 'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓',
       'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓',
       'IT永翔海外仓','CN易速达:易速达美东GA仓', 
       # '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓', 
                               '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国芝加哥S06仓',
        '7天日均', '15天日均', '30天日均', '断货时间1',
       '断货总天数1', '损失销量1', '断货开始天数1',
       '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2',
       '断货时间3', '断货总天数3', '损失销量3', '断货开始天数3'
                    ]

send_goods_ca = send_goods_ca[select_columns_ca]

In [10]:
select_columns2 = ['补货', '借调', 
                  'FBA在库可售天数', 'FBA总可售天数', '预估在库日均', '首次断货前可售天数', 
                  'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre实际可发货数', 
                  'air_pre实际可发货数', 'sea_pre实际可发货数',                                                                       
                  'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数']

key_cols = ['店铺-站点', 'MSKU']

def prepare_period_df(df, suffix):
    df = df[key_cols + select_columns2].copy()
    rename_dict = {value: f'{value}_{suffix}' for value in select_columns2}
    return df.rename(columns=rename_dict)

send_goods15 = prepare_period_df(send_goods15, '15')
send_goods30 = prepare_period_df(send_goods30, '30')
send_goods60 = prepare_period_df(send_goods60, '60')

In [11]:
# shipment_receive_total_no_merge = (
#     shipment_receive_total_no
#     .groupby(key_cols, as_index=False)['差异量']
#     .sum()
# )

# ret = ret.merge(
#     shipment_receive_total_no_merge,
#     how='left',
#     on=key_cols,
#     validate='many_to_one'
# )

In [12]:
def merge_with_period(base_df, period_df, suffix):
    ret = base_df.merge(period_df, on=key_cols, how='left', validate='one_to_one')
    ret['在库天数差'] = ret['FBA在库可售天数'] - ret[f'FBA在库可售天数_{suffix}']
    ret['在库日均比'] = (ret['预估在库日均'] / ret[f'预估在库日均_{suffix}']).fillna(0).round(2)
    
    shipment_receive_total_no_merge = (
    shipment_receive_total_no
    .groupby(key_cols, as_index=False)['差异量']
    .sum()
    )
    
    ret = ret.merge(
        shipment_receive_total_no_merge,
        how='left',
        on=key_cols,
        validate='many_to_one'
    )
    # ret = ret.merge(shipment_receive_total_no[['店铺-站点', 'MSKU', '差异量']], how='left', on=key_cols, validate='one_to_one')
    
    ret['差异量'] = ret['差异量'].fillna(0).astype(int)
    return ret

ret_df60 = merge_with_period(send_goods, send_goods60, '60')
ret_df30 = merge_with_period(send_goods, send_goods30, '30')
ret_df15 = merge_with_period(send_goods, send_goods15, '15')

ret_df_ca = pd.merge(left=send_goods_ca, right=shipment_receive_total_no[['店铺-站点', 'MSKU', '差异量']], how='left', on=['店铺-站点', 'MSKU'])
ret_df_ca['差异量'] = ret_df_ca['差异量'].fillna(0).astype(int)

ret_df_ca = pd.merge(left=ret_df_ca, right=inspection_excluded_total, how='left', on=['店铺-站点', 'MSKU'])
ret_df_ca['不计入在途的查验量'] = ret_df_ca['不计入在途的查验量'].fillna(0).astype(int)

In [13]:
# shipment_receive_total_no.loc[
#     shipment_receive_total_no.duplicated(subset=key_cols, keep=False),
#     ['店铺-站点', 'MSKU', '差异量']
# ].sort_values(key_cols)

In [14]:
ret_df30.columns

Index(['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位',
       ...
       'sea_pre_30', 'dhl_pre实际可发货数_30', 'air_pre实际可发货数_30',
       'sea_pre实际可发货数_30', 'dhl_pre缺货数_30', 'air_pre缺货数_30', 'sea_pre缺货数_30',
       '在库天数差', '在库日均比', '差异量'],
      dtype='object', length=104)

In [15]:
ret_columns15 = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位','总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
       '补货', '借调', '补货_15','借调_15', 
       'FBA在库可售天数', 'FBA总可售天数', 'FBA在库可售天数_15', 'FBA总可售天数_15', 
       '预估在库日均', '预估在库日均_15', '在库天数差', '在库日均比','首次断货前可售天数', '首次断货前可售天数_15',
       'FBA库存', '已出运', 'FBA预占','FBA在途', '差异量',
       'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre_15', 'air_pre_15', 'sea_pre_15', 
       'dhl_pre实际可发货数', 'air_pre实际可发货数','sea_pre实际可发货数', 'dhl_pre实际可发货数_15', 'air_pre实际可发货数_15', 'sea_pre实际可发货数_15', 
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 'dhl_pre缺货数_15','air_pre缺货数_15', 'sea_pre缺货数_15',
       # '预占总数', '快递_预占', '空运_预占', '海运_预占', 
       '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占', 
       'CA', 'DE','JP', 'UK', 'US', 
       'TK本地仓', '共享', '本地-在途', '已下单数量','已生产未发货',
       'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓','JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓','元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
       # '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
                           '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国芝加哥S06仓',
       '7天日均', '15天日均', '30天日均', 
       '断货时间1', '断货总天数1', '损失销量1','断货开始天数1', 
                 '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', 
                 '断货时间3', '断货总天数3','损失销量3', '断货开始天数3'
       ]

# # 指定要添加颜色的列
# ret_columns15_color = {'店铺-站点': 'FFD7E3BD', '站点': 'FFD7E3BD', '负责人': 'FFD7E3BD', 'Listing': 'FFD7E3BD', 
#                 '款式': 'FFD7E3BD', 'MSKU': 'FFD7E3BD', '积加SKU': 'FFD7E3BD', 'FNSKU': 'FFD7E3BD', '仓储类型': 'FFD7E3BD','规模定位': 'FFD7E3BD', 
#                 '需求定位': 'FFD7E3BD', '发货定位': 'FFD7E3BD', '备货定位': 'FFD7E3BD', 
#                 '补货': 'FFFDE9D9', '借调': 'FFFDE9D9', '补货_15': 'FFFCD5B4', '借调_15': 'FFFCD5B4', 
#                 'FBA在库可售天数': 'FFDAEEF3', 'FBA总可售天数': 'FFDAEEF3', 'FBA在库可售天数_15': 'FFB7DEE8', 'FBA总可售天数_15': 'FFB7DEE8', 
#                 '预估在库日均': 'FFE4DFEC', '预估在库日均_15': 'FFCCC0DA', '在库天数差': 'FFD7E3BD', '在库日均比': 'FFA9D18E','首次断货前可售天数': 'FFF2DCDB', '首次断货前可售天数_15': 'FFE6B8B7',
#                 'FBA库存': 'FFB8CCE4', '已出运': 'FFB8CCE4', 'FBA预占': 'FFB8CCE4','FBA在途': 'FFB8CCE4', '差异量': 'FFE6B8B7',
#                 'dhl_pre': 'FFFDE9D9', 'air_pre': 'FFFDE9D9', 'sea_pre': 'FFFDE9D9', 'dhl_pre_15': 'FFFCD5B4', 'air_pre_15': 'FFFCD5B4', 'sea_pre_15': 'FFFCD5B4', 
#                 'dhl_pre实际可发货数': 'FFDAEEF3', 'air_pre实际可发货数': 'FFDAEEF3','sea_pre实际可发货数': 'FFDAEEF3', 'dhl_pre实际可发货数_15': 'FFB7DEE8', 'air_pre实际可发货数_15': 'FFB7DEE8', 'sea_pre实际可发货数_15': 'FFB7DEE8', 
#                 'dhl_pre缺货数': 'FFE4DFEC', 'air_pre缺货数': 'FFE4DFEC', 'sea_pre缺货数': 'FFE4DFEC', 'dhl_pre缺货数_15': 'FFE4DFEC','air_pre缺货数_15': 'FFE4DFEC', 'sea_pre缺货数_15': 'FFE4DFEC',
#                 'CA': 'FFD7E3BD', 'DE': 'FFD7E3BD','JP': 'FFD7E3BD', 'UK': 'FFD7E3BD', 'US': 'FFD7E3BD', '共享': 'FFD7E3BD', '本地-在途': 'FFF2DCDB', '已下单数量': 'FFE6B8B7',
#                 'CA仓搜海外仓': 'FFB8CCE4', 'DE商易海外仓': 'FFB8CCE4', 'DE延讯海外仓': 'FFB8CCE4','JP永翔海外仓': 'FFB8CCE4', 'UK商易海外仓': 'FFB8CCE4', 'UK延讯海外仓': 'FFB8CCE4', 'US商易海外仓': 'FFB8CCE4', 'US易速达海外仓': 'FFB8CCE4', '九方欧洲海外仓': 'FFB8CCE4','元坤海外仓': 'FFB8CCE4', 
#                 '7天日均': 'FFFCD5B4', '15天日均': 'FFFCD5B4', '30天日均': 'FFFCD5B4', 
#                 '断货时间1': 'FFD7E3BD', '断货总天数1': 'FFD7E3BD', '损失销量1': 'FFD7E3BD','断货开始天数1': 'FFD7E3BD', '断货时间2': 'FFD7E3BD', '断货总天数2': 'FFD7E3BD', '损失销量2': 'FFD7E3BD', '断货开始天数2': 'FFD7E3BD', '断货时间3': 'FFD7E3BD', '断货总天数3': 'FFD7E3BD','损失销量3': 'FFD7E3BD', '断货开始天数3': 'FFD7E3BD'
#                       }

In [16]:
ret_columns30 = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
       '补货', '借调', '补货_30','借调_30', 
       'FBA在库可售天数', 'FBA总可售天数', 'FBA在库可售天数_30', 'FBA总可售天数_30', 
       '预估在库日均', '预估在库日均_30', '在库天数差', '在库日均比','首次断货前可售天数', '首次断货前可售天数_30',
       'FBA库存', '已出运', 'FBA预占','FBA在途','差异量',
       'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre_30', 'air_pre_30', 'sea_pre_30', 
       'dhl_pre实际可发货数', 'air_pre实际可发货数','sea_pre实际可发货数', 'dhl_pre实际可发货数_30', 'air_pre实际可发货数_30', 'sea_pre实际可发货数_30', 
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 'dhl_pre缺货数_30','air_pre缺货数_30', 'sea_pre缺货数_30',
       # '预占总数', '快递_预占', '空运_预占', '海运_预占', 
       '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占', 
       'CA', 'DE','JP', 'UK', 'US', 
       'TK本地仓', '共享', '本地-在途', '已下单数量','已生产未发货',
       'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓','JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓','元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
       # '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
                           '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国芝加哥S06仓',
       '7天日均', '15天日均', '30天日均', 
       '断货时间1', '断货总天数1', '损失销量1','断货开始天数1', 
                 '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', 
                 '断货时间3', '断货总天数3','损失销量3', '断货开始天数3'
                ]


# ret_columns30_color = {'店铺-站点': 'FFD7E3BD', '站点': 'FFD7E3BD', '负责人': 'FFD7E3BD', 'Listing': 'FFD7E3BD', 
#                 '款式': 'FFD7E3BD', 'MSKU': 'FFD7E3BD', '积加SKU': 'FFD7E3BD', 'FNSKU': 'FFD7E3BD', '仓储类型': 'FFD7E3BD','规模定位': 'FFD7E3BD', 
#                 '需求定位': 'FFD7E3BD', '发货定位': 'FFD7E3BD', '备货定位': 'FFD7E3BD', 
#                 '补货': 'FFFDE9D9', '借调': 'FFFDE9D9', '补货_30':'FFFCD5B4', '借调_30': 'FFFCD5B4', 
#                 'FBA在库可售天数': 'FFDAEEF3', 'FBA总可售天数': 'FFDAEEF3', 'FBA在库可售天数_30': 'FFB7DEE8', 'FBA总可售天数_30': 'FFB7DEE8', 
#                 '预估在库日均': 'FFE4DFEC', '预估在库日均_30': 'FFCCC0DA', '在库天数差': 'FFD7E3BD', '在库日均比': 'FFA9D18E','首次断货前可售天数': 'FFF2DCDB', '首次断货前可售天数_30': 'FFE6B8B7',
#                 'FBA库存': 'FFB8CCE4', '已出运': 'FFB8CCE4', 'FBA预占': 'FFB8CCE4','FBA在途': 'FFB8CCE4', '差异量': 'FFE6B8B7',
#                 'dhl_pre': 'FFFDE9D9', 'air_pre': 'FFFDE9D9', 'sea_pre': 'FFFDE9D9', 'dhl_pre_30': 'FFFCD5B4', 'air_pre_30': 'FFFCD5B4', 'sea_pre_30': 'FFFCD5B4', 
#                 'dhl_pre实际可发货数': 'FFDAEEF3', 'air_pre实际可发货数': 'FFDAEEF3','sea_pre实际可发货数': 'FFDAEEF3', 'dhl_pre实际可发货数_30': 'FFB7DEE8', 'air_pre实际可发货数_30': 'FFB7DEE8', 'sea_pre实际可发货数_30': 'FFB7DEE8', 
#                 'dhl_pre缺货数': 'FFE4DFEC', 'air_pre缺货数': 'FFE4DFEC', 'sea_pre缺货数': 'FFE4DFEC', 'dhl_pre缺货数_30': 'FFE4DFEC','air_pre缺货数_30': 'FFE4DFEC', 'sea_pre缺货数_30': 'FFE4DFEC',
#                 'CA': 'FFD7E3BD', 'DE': 'FFD7E3BD','JP': 'FFD7E3BD', 'UK': 'FFD7E3BD', 'US': 'FFD7E3BD', '共享': 'FFD7E3BD', '本地-在途': 'FFF2DCDB', '已下单数量': 'FFE6B8B7',
#                 'CA仓搜海外仓': 'FFB8CCE4', 'DE商易海外仓': 'FFB8CCE4', 'DE延讯海外仓': 'FFB8CCE4','JP永翔海外仓': 'FFB8CCE4', 'UK商易海外仓': 'FFB8CCE4', 'UK延讯海外仓': 'FFB8CCE4', 'US商易海外仓': 'FFB8CCE4', 'US易速达海外仓': 'FFB8CCE4', '九方欧洲海外仓': 'FFB8CCE4','元坤海外仓': 'FFB8CCE4', 
#                 '7天日均': 'FFFCD5B4', '15天日均': 'FFFCD5B4', '30天日均': 'FFFCD5B4', 
#                 '断货时间1': 'FFD7E3BD', '断货总天数1': 'FFD7E3BD', '损失销量1': 'FFD7E3BD','断货开始天数1': 'FFD7E3BD', '断货时间2': 'FFD7E3BD', '断货总天数2': 'FFD7E3BD', '损失销量2': 'FFD7E3BD', '断货开始天数2': 'FFD7E3BD', '断货时间3': 'FFD7E3BD', '断货总天数3': 'FFD7E3BD','损失销量3': 'FFD7E3BD', '断货开始天数3': 'FFD7E3BD'
#                       }

In [17]:
ret_columns60 = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数', '快递打包时间', '空运打包时间', '海运打包时间',
       '补货', '借调', '补货_60','借调_60', 
       'FBA在库可售天数', 'FBA总可售天数', 'FBA在库可售天数_60', 'FBA总可售天数_60', 
       '预估在库日均', '预估在库日均_60', '在库天数差', '在库日均比','首次断货前可售天数', '首次断货前可售天数_60',
       'FBA库存', '已出运', 'FBA预占','FBA在途','差异量',
       'dhl_pre', 'air_pre', 'sea_pre', 'dhl_pre_60', 'air_pre_60', 'sea_pre_60', 
       'dhl_pre实际可发货数', 'air_pre实际可发货数','sea_pre实际可发货数', 'dhl_pre实际可发货数_60', 'air_pre实际可发货数_60', 'sea_pre实际可发货数_60', 
       'dhl_pre缺货数', 'air_pre缺货数', 'sea_pre缺货数', 'dhl_pre缺货数_60','air_pre缺货数_60', 'sea_pre缺货数_60',
       # '预占总数', '快递_预占', '空运_预占', '海运_预占', 
       '预占总数', 'FBA自提物流_预占','快递_预占', '空运_预占', '海运_预占', 
       'CA', 'DE','JP', 'UK', 'US', 
       'TK本地仓', '共享', '本地-在途', '已下单数量','已生产未发货',
       'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓','JP永翔海外仓', 'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓','元坤海外仓', 'IT永翔海外仓', 'CN易速达:易速达美东GA仓',
       # '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓',
                           '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国芝加哥S06仓',
       '7天日均', '15天日均', '30天日均', 
       '断货时间1', '断货总天数1', '损失销量1','断货开始天数1', 
                 '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2', 
                 '断货时间3', '断货总天数3','损失销量3', '断货开始天数3'
                ]

In [18]:
ret_columns_ca = ['店铺-站点', '站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', 'FNSKU', '仓储类型',
       '规模定位', '需求定位', '发货定位', '备货定位', '总发货天数', '快递在途天数', '空运在途天数', '海运在途天数',
       '快递打包时间', '空运打包时间', '海运打包时间', '补货', '借调', 'FBA在库可售天数', 'FBA总可售天数',
       '本地库存可售天数', '总库存可售天数', '预估在库日均', '断货风险总天数', '断货总损失销量', '首次断货前可售天数',
       'FBA库存', '已出运', 'FBA预占', 'FBA在途','差异量','差异量-已计入可用库存','不计入在途的查验量', 'dhl_pre', 'air_pre', 'sea_pre',
       'dhl_pre实际可发货数', 'air_pre实际可发货数', 'sea_pre实际可发货数', 'dhl_pre缺货数',
       'air_pre缺货数', 'sea_pre缺货数', '预占总数', '快递_预占', '空运_预占', '海运_预占',
       'FBA自提物流_预占', 'CA', 'DE', 'JP', 'UK', 'US', 'TK本地仓', '共享', '本地-在途',
       '已下单数量', '已生产未发货', 'CA仓搜海外仓', 'DE商易海外仓', 'DE延讯海外仓', 'JP永翔海外仓',
       'UK商易海外仓', 'UK延讯海外仓', 'US商易海外仓', 'US易速达海外仓', '九方欧洲海外仓', '元坤海外仓',
       'IT永翔海外仓', 'CN易速达:易速达美东GA仓', 
       # '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国达拉斯C1仓', '顺丰SF:美国芝加哥C1仓', 
                            '顺丰SF:美国特拉华S2仓', '顺丰SF:美国洛杉矶S5仓', '顺丰SF:美国芝加哥S06仓',
       '7天日均', '15天日均', '30天日均', '断货时间1',
       '断货总天数1', '损失销量1', '断货开始天数1', 
                  '断货时间2', '断货总天数2', '损失销量2', '断货开始天数2',
       '断货时间3', '断货总天数3', '损失销量3', '断货开始天数3'
                 ]

In [19]:
with pd.ExcelWriter(f'../程序建议发货列表/发货列表{parms}-new-7.2.xlsx', engine='openpyxl') as writer:
    ret_df_ca.to_excel(writer, sheet_name='预估', index=False, columns=ret_columns_ca)
    ret_df15.to_excel(writer, sheet_name='预估VS15', index=False, columns=ret_columns15)
    ret_df30.to_excel(writer, sheet_name='预估VS30', index=False, columns=ret_columns30)
    ret_df60.to_excel(writer, sheet_name='预估VS60', index=False, columns=ret_columns60)
    shipment_receive_detail.to_excel(writer, sheet_name='接收中-货件详情', index=False)
    future_records_detail.to_excel(writer, sheet_name='预计到仓日期早于当天日期的记录-详情', index=False)
    future_records_total.to_excel(writer, sheet_name='预计到仓日期早于当天日期的记录-汇总', index=False)
    inspection_excluded_df.to_excel(writer, sheet_name='不计入在途的查验货件详情', index=False)